# Dispersione del residuo, esemplare per esemplare

Notebook autonomo, fuori dalla catena numerata: non scrive nulla che gli altri
notebook leggano e non modifica `funzioni.py`.

Serve a rispondere a una domanda sola: **quanto della variazione del residuo si
spiega con quale cuscinetto e' montato, e quanto con la sua classe di guasto?**

Il notebook 03 salva per ogni esemplare media, mediana e quartili, ma non la
deviazione standard, quindi la scomposizione della varianza si poteva solo
stimare. Qui il riferimento viene riaddestrato una volta sola --- 2560 frame
sani, 500 epoche, seme 0, esattamente la replica letterale --- e i residui
frame per frame restano in memoria, cosi tutti i conti sono esatti.

Costa un addestramento solo, non i dieci del notebook 03.

In [ ]:
!apt-get -qq update && apt-get -qq install -y unrar
!pip -q install requests scipy scikit-learn

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO = 'https://github.com/matpaol/MacchineEdAzionamentiExam'
possibili = ['.', '..', '../codice',
             '/content/drive/MyDrive/MacchineEdAzionamentiExam',
             '/content/MacchineEdAzionamentiExam']
percorso_codice = None
for c in possibili:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break
if percorso_codice is None:
    subprocess.run(['git', 'clone', '-q', REPO, '/content/MacchineEdAzionamentiExam'],
                   check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam'
sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='extra_dispersione')
dev = f.dispositivo()
seme = 0

print('codice da', percorso_codice, '| dispositivo', dev)

## I dati della replica

Identici al notebook 03: diciassette cuscinetti, un solo regime, quattro
segmenti da un secondo per registrazione, frame da 2560 campioni. Le due righe
che scelgono i 2560 frame sani usano lo stesso seme, quindi selezionano
esattamente gli stessi frame.

In [ ]:
cuscinetti = config.CUSCINETTI_PAPER
regime = config.REGIME_PRINCIPALE

f.estrai_misure(cuscinetti, P['raw'], P['estratti'])
inv = f.inventario(cuscinetti, P['estratti'], regimi=[regime]).reset_index(drop=True)
segmenti, anagrafica = f.costruisci_segmenti(inv, segmenti_per_registrazione=4)

lunghezza_frame = f.lunghezza_frame(config.REGIMI[regime]['rpm'])
frame_per_segmento = segmenti.shape[1] // lunghezza_frame
frame = segmenti.reshape(-1, lunghezza_frame)

classe_del_frame = np.repeat(anagrafica['classe'].values, frame_per_segmento)
cuscinetto_del_frame = np.repeat(anagrafica['cuscinetto'].values, frame_per_segmento)

rng = np.random.default_rng(seme)
scelti = rng.choice(np.flatnonzero(classe_del_frame == 0), size=2560, replace=False)
rng.shuffle(scelti)
frame_train, frame_val = frame[scelti[:2048]], frame[scelti[2048:]]

print(len(segmenti), 'segmenti |', len(frame), 'frame da', lunghezza_frame, 'campioni')
print('frame sani per il DAE:', len(scelti))

## L'autoencoder di riferimento

La replica letterale: SELU anche sull'uscita, nessuna scalatura, 500 epoche
fisse, nessun arresto anticipato. E' la prima riga della tabella del
Capitolo 7, e i numeri che escono da qui devono coincidere con quelli.

In [ ]:
modello, curva_train, curva_val = f.addestra_dae(
    frame_train, frame_val, uscita_selu=True, epoche=500, lotto=256,
    passo=3e-4, pazienza=None, seme=seme, dev=dev, stampa_ogni=250)

residui = f.calcola_residui(modello, frame, dev=dev)
mse = f.mse_per_frame(residui)

per_classe = [float(np.mean(mse[classe_del_frame == c])) for c in (0, 1, 2)]
print()
print('controllo contro il Capitolo 7, riga "Replica letterale":')
print('   residuo sani     {:.4f}   atteso 0,0801'.format(per_classe[0]))
print('   rapporto esterno {:.3f}    atteso 1,037'.format(per_classe[1] / per_classe[0]))
print('   rapporto interno {:.3f}    atteso 1,142'.format(per_classe[2] / per_classe[0]))

## Statistiche per esemplare, con la deviazione standard

Le stesse colonne che salva `residuo_per_cuscinetto`, piu la deviazione
standard e il coefficiente di variazione, cioe la deviazione standard divisa
per la media. Quest'ultimo dice quanto e' compatto il residuo di un cuscinetto
rispetto al suo stesso livello, e permette di confrontare fra loro esemplari
che stanno su livelli diversi.

In [ ]:
righe = []
for nome in sorted(set(cuscinetto_del_frame)):
    v = mse[cuscinetto_del_frame == nome].astype(np.float64)
    righe.append({
        'cuscinetto': nome,
        'classe': config.NOMI_CLASSI[config.CLASSE_DI[nome]],
        'n': int(v.size),
        'media': float(np.mean(v)),
        'std': float(np.std(v, ddof=1)),
        'cv_pct': float(100 * np.std(v, ddof=1) / np.mean(v)),
        'mediana': float(np.median(v)),
        'q1': float(np.percentile(v, 25)),
        'q3': float(np.percentile(v, 75)),
        'minimo': float(np.min(v)),
        'massimo': float(np.max(v)),
    })

per_cuscinetto = pd.DataFrame(righe).sort_values('media').reset_index(drop=True)
print(per_cuscinetto[['cuscinetto', 'classe', 'n', 'media', 'std', 'cv_pct',
                      'mediana', 'q1', 'q3']].round(5).to_string(index=False))
print()
print('deviazione standard dentro un cuscinetto: da {:.5f} a {:.5f}, mediana {:.5f}'.format(
    per_cuscinetto['std'].min(), per_cuscinetto['std'].max(),
    per_cuscinetto['std'].median()))
print('coefficiente di variazione: da {:.1f}% a {:.1f}%, mediana {:.1f}%'.format(
    per_cuscinetto['cv_pct'].min(), per_cuscinetto['cv_pct'].max(),
    per_cuscinetto['cv_pct'].median()))

## Quanta parte della variazione spiega l'esemplare, e quanta la classe

La domanda si risponde scomponendo la varianza totale dei residui in due
addendi: quella **fra** i gruppi e quella **dentro** i gruppi. Il rapporto fra
la prima e il totale dice quale frazione della variazione osservata e'
spiegata dal criterio con cui si sono formati i gruppi.

Il conto viene fatto due volte con la stessa formula: una raggruppando per
cuscinetto, diciassette gruppi, e una raggruppando per classe di guasto, tre
gruppi. Il confronto fra i due numeri e' il risultato che interessa.

In [ ]:
def scomponi(valori, etichette):
    """Frazione della varianza totale spiegata dal raggruppamento."""
    valori = np.asarray(valori, dtype=np.float64)
    etichette = np.asarray(etichette)
    media_generale = valori.mean()
    fra, dentro = 0.0, 0.0
    for g in np.unique(etichette):
        v = valori[etichette == g]
        fra += v.size * (v.mean() - media_generale) ** 2
        dentro += float(((v - v.mean()) ** 2).sum())
    return {'gruppi': int(np.unique(etichette).size),
            'devianza_fra': fra, 'devianza_dentro': dentro,
            'quota_spiegata': fra / (fra + dentro)}


per_esemplare = scomponi(mse, cuscinetto_del_frame)
per_classe_dec = scomponi(mse, classe_del_frame)

print('raggruppando per CUSCINETTO ({} gruppi): spiega il {:.1f}% della variazione'.format(
    per_esemplare['gruppi'], 100 * per_esemplare['quota_spiegata']))
print('raggruppando per CLASSE     ({} gruppi): spiega il {:.1f}% della variazione'.format(
    per_classe_dec['gruppi'], 100 * per_classe_dec['quota_spiegata']))
print()
print('rapporto fra le due quote: {:.1f}'.format(
    per_esemplare['quota_spiegata'] / per_classe_dec['quota_spiegata']))
print()
print('deviazione standard fra le medie dei 17 esemplari: {:.5f}'.format(
    per_cuscinetto['media'].std(ddof=1)))
print('deviazione standard tipica dentro un esemplare   : {:.5f}'.format(
    per_cuscinetto['std'].median()))

## Quanto sbagliava la stima ricavata dai quartili

Prima di questo notebook la deviazione standard era stimata dai quartili con
la formula valida per una distribuzione normale, cioe ampiezza interquartile
divisa per 1,349. La cella confronta stima e valore vero, esemplare per
esemplare: serve a sapere se quella scorciatoia fosse accettabile.

In [ ]:
confronto_stima = per_cuscinetto[['cuscinetto', 'classe', 'std']].copy()
confronto_stima['std_stimata'] = (per_cuscinetto['q3'] - per_cuscinetto['q1']) / 1.349
confronto_stima['errore_pct'] = 100 * (confronto_stima['std_stimata']
                                       / confronto_stima['std'] - 1)
print(confronto_stima.round(5).to_string(index=False))
print()
print('errore della stima: da {:+.1f}% a {:+.1f}%, in valore assoluto {:.1f}% in media'.format(
    confronto_stima['errore_pct'].min(), confronto_stima['errore_pct'].max(),
    confronto_stima['errore_pct'].abs().mean()))

## I sei cuscinetti sani

E' il confronto piu pulito che i dati permettano: sei esemplari dello stesso
costruttore, con la stessa geometria e senza alcun danno. Qualunque differenza
fra i loro residui non puo essere attribuita a un guasto.

In [ ]:
sani = per_cuscinetto[per_cuscinetto['classe'] == 'normale'].copy()
sani['ore_rodaggio'] = sani['cuscinetto'].map(
    {n: config.ANAGRAFICA[n]['ore'] for n in config.CUSCINETTI_PER_CLASSE[0]})
sani['produttore'] = sani['cuscinetto'].map(
    {n: config.ANAGRAFICA[n]['produttore'] for n in config.CUSCINETTI_PER_CLASSE[0]})
print(sani[['cuscinetto', 'produttore', 'ore_rodaggio', 'media', 'std', 'cv_pct']]
      .round(5).to_string(index=False))
print()
print('rapporto fra il residuo piu alto e il piu basso, fra i soli sani: {:.3f}'.format(
    sani['media'].max() / sani['media'].min()))
print('per confronto, rapporto medio guasti su sani: {:.3f}'.format(
    np.mean(mse[classe_del_frame > 0]) / np.mean(mse[classe_del_frame == 0])))
print()
correlazione = np.corrcoef(sani['ore_rodaggio'], sani['media'])[0, 1]
print('correlazione fra ore di rodaggio e residuo, sui sei sani: {:.3f}'.format(correlazione))
print('(sei punti soltanto: serve a escludere una tendenza netta, non a stabilirne una)')

## L'estensione del danno spiega l'ordine dei residui?

Le schede dichiarano per ogni cuscinetto danneggiato l'estensione del difetto
secondo la norma VDI 3832. Se il residuo misurasse la gravita del guasto, i
gruppi dovrebbero risultare ordinati.

In [ ]:
reali = per_cuscinetto[per_cuscinetto['classe'] != 'normale'].copy()
reali['estensione'] = [config.ANAGRAFICA[n]['estensione'] for n in reali['cuscinetto']]

print(reali.groupby('estensione').agg(
    cuscinetti=('cuscinetto', 'count'),
    residuo_medio=('media', 'mean'),
    elenco=('cuscinetto', lambda s: ' '.join(s))).round(5).to_string())
print()
print('correlazione fra estensione dichiarata e residuo: {:.3f}'.format(
    np.corrcoef(reali['estensione'], reali['media'])[0, 1]))
print('(i gruppi sono molto sbilanciati: il confronto esclude una tendenza netta,')
print(' non ne stabilisce una)')

## Figura e salvataggi

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(12, 4.4))

colori = [f.COLORI[c] for c in per_cuscinetto['classe']]
posizioni = np.arange(len(per_cuscinetto))
assi[0].errorbar(per_cuscinetto['media'], posizioni,
                 xerr=per_cuscinetto['std'], fmt='none',
                 ecolor=f.COLORI['neutro'], elinewidth=1, capsize=2)
assi[0].scatter(per_cuscinetto['media'], posizioni, c=colori, s=42, zorder=3)
assi[0].set_yticks(posizioni)
assi[0].set_yticklabels(per_cuscinetto['cuscinetto'], fontsize=8)
assi[0].set_xlabel('residuo medio, con una deviazione standard (A$^2$)')
assi[0].set_title('Ogni esemplare con la sua dispersione interna', fontsize=10)
for cl in config.NOMI_CLASSI:
    assi[0].plot([], [], 'o', color=f.COLORI[cl], label=cl)
assi[0].legend(fontsize=7)

quote = [100 * per_esemplare['quota_spiegata'], 100 * per_classe_dec['quota_spiegata']]
assi[1].bar(['per cuscinetto', 'per classe'], quote,
            color=[f.COLORI['nostro'], f.COLORI['neutro']], width=0.5)
for x, v in enumerate(quote):
    assi[1].text(x, v + 1.5, '{:.1f}%'.format(v).replace('.', ','),
                 ha='center', fontsize=9)
assi[1].set_ylim(0, 100)
assi[1].set_ylabel('variazione del residuo spiegata (%)')
assi[1].set_title('Che cosa spiega il residuo', fontsize=10)

f.salva_figura(fig, 'dispersione_esemplari', P['figure'])
plt.show()

f.salva_tabella(per_cuscinetto, 'dispersione_per_cuscinetto', P['tabelle'])
f.salva_tabella(confronto_stima, 'confronto_stima_quartili', P['tabelle'])

In [ ]:
def tabella_latex(df, colonne, intestazioni, decimali=5):
    def fmt(v):
        if isinstance(v, (float, np.floating)):
            return ('{:.' + str(decimali) + 'f}').format(v).replace('.', ',')
        return str(v)
    print('\\begin{tabular}{' + 'l' * 2 + 'c' * (len(colonne) - 2) + '}')
    print('\t\\toprule')
    print('\t' + ' & '.join(intestazioni) + ' \\\\')
    print('\t\\midrule')
    for _, r in df.iterrows():
        print('\t' + ' & '.join(fmt(r[c]) for c in colonne) + ' \\\\')
    print('\t\\bottomrule')
    print('\\end{tabular}')


da_stampare = per_cuscinetto.copy()
da_stampare['cv_pct'] = da_stampare['cv_pct'].map(lambda v: '{:.1f}'.format(v).replace('.', ','))

print('TABELLA PER IL CAPITOLO 7')
print('=' * 60)
tabella_latex(da_stampare,
              ['cuscinetto', 'classe', 'media', 'std', 'cv_pct'],
              ['cuscinetto', 'classe', 'residuo medio', 'dev.\\ standard',
               'coeff.\\ di variazione (\\%)'], decimali=4)
print()
print('FRASI CON I NUMERI ESATTI, DA RIPORTARE NEL TESTO')
print('=' * 60)
print('deviazione standard tipica dentro un esemplare: {:.5f}, pari al {:.1f}% del suo residuo medio'
      .format(per_cuscinetto['std'].median(), per_cuscinetto['cv_pct'].median()))
print('deviazione standard fra le medie dei diciassette esemplari: {:.5f}'
      .format(per_cuscinetto['media'].std(ddof=1)))
print('l esemplare spiega il {:.1f}% della variazione del residuo, la classe il {:.1f}%'
      .format(100 * per_esemplare['quota_spiegata'],
              100 * per_classe_dec['quota_spiegata']))
print('fra i soli sei cuscinetti sani il residuo varia di un fattore {:.2f}'
      .format(sani['media'].max() / sani['media'].min()))